# Бинаризация и масштабирование — только подготовка данных

**Цель:** из очищенного файла `data_clean.parquet` создать четыре числовые таблицы для обучения моделей из ДЗ 8 (включая CatBoost).

- **2022** — train, **2023** — test; `is_out_next_year` — целевая переменная, не входящая в признаки.
- В каждой выборке — одинаковые признаки с масштабированием и без масштабирования.
- Target Encoding, отбор высококоррелированных One-Hot признаков, винзоризация, заполнение медианой и масштабирование **обучаются на 2022 году**; к 2023 году применяются те же параметры.
- **Балансировка классов и обучение моделей здесь не выполняются.** Для SMOTE требуется делать балансировку только на обучении (и внутри фолдов при кросс-валидации).

Путь к исходному файлу и папка результата настраиваются ниже. Если исходный файл лежит по другому пути, поправьте `DATA_PATH`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, StandardScaler, MinMaxScaler, RobustScaler
from packaging.version import Version
import joblib

## 1. Загрузка очищенных данных

In [2]:
DATA_PATH = Path('../data/data_clean.parquet')
OUT = Path('../data')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Файл {DATA_PATH} не найден. Укажите правильный путь в DATA_PATH.'
    )

OUT.mkdir(parents=True, exist_ok=True)
df_pedagog_all = pd.read_parquet(DATA_PATH)
print('Исходный файл:', DATA_PATH, '| строк:', len(df_pedagog_all))

Исходный файл: ..\data\data_clean.parquet | строк: 417149


## 2. Целевая переменная и очищенные значения

In [3]:
# Целевая переменная и очищенные числовые признаки
df_pedagog_all['is_out_next_year'] = df_pedagog_all['target']
df_pedagog_all = df_pedagog_all.drop(columns=['target'])

clean_cols = {
    'experience_age_all_clean': 'experience_age_all',
    'experience_age_pedagogic_clean': 'experience_age_pedagogic',
    'experience_age_other_clean': 'experience_age_other',
    'category_date_clean': 'category_date',
    'category_due_clean': 'category_due',
}

for clean_col, col in clean_cols.items():
    if clean_col in df_pedagog_all.columns:
        df_pedagog_all[col] = df_pedagog_all[clean_col]

## 3. Укрупнение категорий и удаление технических полей

In [4]:
RULES = {
    'учитель': [
        'английского языка', 'русского языка', 'французского языка',
        'немецкого языка', 'испанского языка', 'итальянского языка',
        'китайского языка', 'арабского языка', 'родного языка',
        'родной литературы', 'литературы', 'литературного чтения',
        'истории', 'всеобщей истории', 'истории России',
        'обществознания', 'обществознания и естествознания (окружающий мир)',
        'права', 'экономики',
        'основ духовно-нравственной культуры народов России',
        'математики', 'алгебры', 'геометрии', 'физики', 'информатики',
        'математики и информатики', 'в сфере ИКТ', 'естествознания',
        'биологии', 'химии', 'географии',
        'изобразительного искусства', 'музыки', 'технологии',
        'физической культуры', 'основ безопасности жизнедеятельности',
        'начальных классов',
    ],

    'воспитатель': [
        'офицер кадетского класса', 'офицер-воспитатель кадетского класса',
        'по социальной работе',
        'по социализации, воспитанию и безопасности обучающихся',
        'по профориентации',
        'группы полного дня, группы ОВЗ', 'группы полного дня',
        'группы продленного дня', 'группы кратковременного пребывания',
        'группы ОВЗ', 'интерната, группы ОВЗ',
    ],

    'управление': [
        'по контролю качества образования', 'учебной части',
        'по содержанию образования, конвергенции образовательных программ',
        'по комплексной безопасности',
        'по материально-техническому обеспечению',
        'по управлению ресурсами', 'по питанию',
        'по финансово-экономической деятельности', 'по учету контингента',
        'по музейной работе', 'учебно-производственной мастерской',
    ],
}

In [5]:
value_to_group = {}
for group, values in RULES.items():
    for v in values:
        value_to_group[v.strip().lower()] = group


def categorize(value):
    if pd.isna(value):
        return 'не указано'
    v = str(value).strip().lower()
    return value_to_group.get(v, 'прочее')

In [6]:
df_pedagog_all['speciality_cat'] = df_pedagog_all['speciality'].apply(categorize)

df_pedagog_clean = df_pedagog_all.drop_duplicates(
    subset=['id', 'year'], keep='first'
).copy()

# Для моделирования нужны только строки с известной целью
df_pedagog_clean = df_pedagog_clean[
    df_pedagog_clean['year'].isin([2022, 2023]) &
    df_pedagog_clean['is_out_next_year'].notna()
].copy()

df_pedagog_clean['is_out_next_year'] = (
    df_pedagog_clean['is_out_next_year'].astype(int)
)

# Удаляем исходные текстовые и технические столбцы
cols_to_remove = [
    'academic_degree', 'main_position', 'speciality',
    'category_qual', 'subject', 'parallel',
    'experience_age_all_clean',
    'experience_age_pedagogic_clean',
    'experience_age_other_clean',
    'category_date_clean', 'category_due_clean'
]

df_pedagog_clean = df_pedagog_clean.drop(
    columns=[c for c in cols_to_remove if c in df_pedagog_clean.columns]
)

## 4. Кодирование признаков

In [7]:
# One-Hot Encoding
main_ohe = pd.get_dummies(
    df_pedagog_clean['main_position_cat'],
    prefix='main_position'
).astype(int)

main_ohe = main_ohe.rename(columns={
    'main_position_другой': 'main_position_other',
    'main_position_учитель': 'main_position_teacher',
    'main_position_руководитель': 'main_position_head'
})

spec_ohe = pd.get_dummies(
    df_pedagog_clean['speciality_cat'],
    prefix='speciality'
).astype(int)

spec_ohe = spec_ohe.rename(columns={
    'speciality_не указано': 'speciality_not_specified',
    'speciality_учитель': 'speciality_teacher',
    'speciality_воспитатель': 'speciality_educator',
    'speciality_управление': 'speciality_management',
    'speciality_прочее': 'speciality_other'
})

df_pedagog_clean = pd.concat(
    [df_pedagog_clean, main_ohe, spec_ohe], axis=1
)

In [8]:
# Бинарная кодировка: 1 — степень/категория/действующая категория, 0 — нет
# степень и квалификация объединяются в категории 0/1.
df_pedagog_clean['academic_degree_cat'] = (
    df_pedagog_clean['academic_degree_cat']
    .map({'не_найдено': 0, 'кандидат': 1, 'доктор': 1})
)
df_pedagog_clean['category_qual_cat'] = (
    df_pedagog_clean['category_qual_cat']
    .map({'не_найдено': 0, 'первая': 1, 'вторая': 1, 'высшая': 1})
)
df_pedagog_clean['category_active'] = (
    df_pedagog_clean['category_active']
    .map({'не было': 0, 'не активно': 0, 'активно': 1})
)

# Служебные текстовые категории больше не нужны
# Незапланированные значения показываем явно, не заменяем без проверки
for col in ['academic_degree_cat', 'category_qual_cat', 'category_active']:
    if df_pedagog_clean[col].isna().any():
        raise ValueError(f'Найдены неизвестные категории в {col}; проверьте mapping.')

df_pedagog_clean = df_pedagog_clean.drop(columns=['main_position_cat', 'speciality_cat'])

## 5. Числовые преобразования и новые признаки

In [9]:
# Границы винзоризации определяем только по train (2022),
# а затем одинаково применяем к train и test.
train_year_mask = df_pedagog_clean['year'] == 2022
q01, q99 = df_pedagog_clean.loc[
    train_year_mask, 'experience_age_pedagogic'
].quantile([0.01, 0.99])

df_pedagog_clean['experience_age_pedagogic'] = (
    df_pedagog_clean['experience_age_pedagogic'].clip(lower=q01, upper=q99)
)
df_pedagog_clean['n_unique_subjects'] = np.log1p(
    df_pedagog_clean['n_unique_subjects'].clip(lower=0)
)
df_pedagog_clean['experience_age_other'] = np.log1p(
    df_pedagog_clean['experience_age_other'].clip(lower=0)
)

In [10]:
# Дополнительные варианты возраста и стажа
df_pedagog_clean['age_sq'] = (df_pedagog_clean['age'] / 10) ** 2
df_pedagog_clean['ped_exp_sq'] = (
    df_pedagog_clean['experience_age_pedagogic'] / 10
) ** 2

# Информация о наличии и сроках квалификационной категории
df_pedagog_clean['has_category_date'] = (
    df_pedagog_clean['category_date'].notna().astype(int)
)
df_pedagog_clean['has_category_due'] = (
    df_pedagog_clean['category_due'].notna().astype(int)
)

df_pedagog_clean['category_years_since'] = np.where(
    df_pedagog_clean['category_date'].notna(),
    (df_pedagog_clean['year'] - df_pedagog_clean['category_date']).clip(lower=0),
    0
)

df_pedagog_clean['category_years_left'] = np.where(
    df_pedagog_clean['category_due'].notna(),
    (df_pedagog_clean['category_due'] - df_pedagog_clean['year']).clip(-5, 10),
    0
)

# Общие показатели школы
school = df_pedagog_clean.groupby(['year', 'school_num'])

df_pedagog_clean['school_staff_count'] = school['id'].transform('count')
df_pedagog_clean['school_mean_age'] = school['age'].transform('mean')
df_pedagog_clean['school_mean_ped_exp'] = school[
    'experience_age_pedagogic'
].transform('mean')

df_pedagog_clean['age_vs_school'] = (
    df_pedagog_clean['age'] - df_pedagog_clean['school_mean_age']
)
df_pedagog_clean['ped_exp_vs_school'] = (
    df_pedagog_clean['experience_age_pedagogic']
    - df_pedagog_clean['school_mean_ped_exp']
)

## 6. Разделение по годам и отбор признаков

In [11]:
train = df_pedagog_clean.loc[df_pedagog_clean['year'] == 2022].copy()
test = df_pedagog_clean.loc[df_pedagog_clean['year'] == 2023].copy()

y_train = train['is_out_next_year'].astype(int)
y_test = test['is_out_next_year'].astype(int)

# Одни и те же группы признаков используются в обоих вариантах датасетов.
te_cols = ['school_num']
std_cols_final = [
    'category_qual_cat', 'n_unique_subjects', 'n_unique_parallel',
    'experience_age_pedagogic', 'age', 'experience_age_all',
    'age_sq', 'ped_exp_sq', 'school_staff_count', 'school_mean_age',
    'school_mean_ped_exp', 'age_vs_school', 'ped_exp_vs_school',
    'category_years_since', 'category_years_left'
]
minmax_cols_final = ['category_active', 'min_parallel', 'max_parallel']
robust_cols = ['experience_age_other']
base_binary = ['sex', 'class_manager', 'academic_degree_cat']
ohe_cols = [
    col for col in train.columns
    if col.startswith('main_position_') or col.startswith('speciality_')
]

# Отбираем почти дублирующиеся One-Hot признаки по train.
# Визуализацию корреляций переносим в аналитический ноутбук.
corr = train[ohe_cols].corr()
cols_to_drop = []
for i, col1 in enumerate(corr.columns):
    for col2 in corr.columns[i + 1:]:
        if abs(corr.loc[col1, col2]) <= 0.85:
            continue
        if col1.startswith('speciality_') and col2.startswith('speciality_'):
            drop_col = ('speciality_not_specified'
                        if 'speciality_not_specified' in {col1, col2} else col1)
        elif col1.startswith('main_position_') and col2.startswith('main_position_'):
            drop_col = col1
        else:
            continue
        if drop_col not in cols_to_drop:
            cols_to_drop.append(drop_col)

bin_cols_final = [c for c in base_binary if c in train.columns]
bin_cols_final += [c for c in ohe_cols if c not in cols_to_drop]
bin_cols_final += ['has_category_date', 'has_category_due']

model_cols = te_cols + std_cols_final + minmax_cols_final + robust_cols + bin_cols_final
assert len(model_cols) == len(set(model_cols)), 'Повторяющиеся признаки в model_cols'

X_train = train[model_cols].copy()
X_test = test[model_cols].copy()

# Медианы рассчитываются только на train.
for col in std_cols_final + minmax_cols_final + robust_cols:
    median = X_train[col].median()
    if pd.isna(median):
        raise ValueError(f'Столбец {col} целиком пуст в train — нужна отдельная обработка.')
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

for name, X in [('train', X_train), ('test', X_test)]:
    if X.isna().any().any():
        raise ValueError(f'В {name} остались пропуски: {X.columns[X.isna().any()].tolist()}')

print('Train 2022:', X_train.shape, '| test 2023:', X_test.shape)
print('Число удалённых признаков:', len(cols_to_drop))

Train 2022: (17396, 31) | test 2023: (17394, 31)
Число удалённых OHE признаков: 1


## 7. Target Encoding и масштабирование

In [12]:
def make_target_encoder():
    if Version(sklearn_version) >= Version('1.9'):
        return TargetEncoder(
            target_type='binary', smooth='auto',
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        )
    return TargetEncoder(
        target_type='binary', smooth='auto',
        cv=5, shuffle=True, random_state=42
    )

# кодирование + масштабирование.
prep_scaled = ColumnTransformer([
    ('te', make_target_encoder(), te_cols),
    ('std', StandardScaler(), std_cols_final),
    ('minmax', MinMaxScaler(), minmax_cols_final),
    ('robust', RobustScaler(), robust_cols),
    ('bin', 'passthrough', bin_cols_final)
], verbose_feature_names_out=False)

# кодирование без масштабирования.
other_cols = [c for c in model_cols if c != 'school_num']
prep_unscaled = ColumnTransformer([
    ('te', make_target_encoder(), te_cols),
    ('other', 'passthrough', other_cols)
], verbose_feature_names_out=False)


def transform_to_frames(transformer, X_train, X_test, y_train):
    train_array = transformer.fit_transform(X_train, y_train)
    test_array = transformer.transform(X_test)
    names = [
        'school_num_te' if c == 'school_num' else c
        for c in transformer.get_feature_names_out()
    ]
    return (
        pd.DataFrame(train_array, columns=names, index=X_train.index),
        pd.DataFrame(test_array, columns=names, index=X_test.index)
    )

X_train_scaled, X_test_scaled = transform_to_frames(
    prep_scaled, X_train, X_test, y_train
)
X_train_unscaled, X_test_unscaled = transform_to_frames(
    prep_unscaled, X_train, X_test, y_train
)

# Упорядочим одинаково для честного сравнения разных алгоритмов.
X_train_unscaled = X_train_unscaled[X_train_scaled.columns]
X_test_unscaled = X_test_unscaled[X_test_scaled.columns]

for name, X in [('train_scaled', X_train_scaled), ('test_scaled', X_test_scaled),
                ('train_unscaled', X_train_unscaled), ('test_unscaled', X_test_unscaled)]:
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(f'В наборе {name} есть NaN или бесконечные значения.')

print('Признаков после подготовки:', X_train_scaled.shape[1])

Признаков после подготовки: 31


## 8. Сохранение четырёх готовых наборов и преобразователей

In [13]:
# Целевая переменная хранится отдельным столбцом и не участвует в преобразованиях.
datasets = {
    'train_scaled': (X_train_scaled, y_train),
    'test_scaled': (X_test_scaled, y_test),
    'train_unscaled': (X_train_unscaled, y_train),
    'test_unscaled': (X_test_unscaled, y_test),
}
for name, (X, y) in datasets.items():
    result = X.copy()
    result['is_out_next_year'] = y.to_numpy()
    result.to_parquet(OUT / f'{name}.parquet', index=False)
    print(f'{name}.parquet — {result.shape[0]} строк, {result.shape[1]} столбцов')

# Преобразователи пригодятся для применения тех же правил к новым данным.
joblib.dump(prep_scaled, OUT / 'preprocessor_scaled.joblib')
joblib.dump(prep_unscaled, OUT / 'preprocessor_unscaled.joblib')


train_scaled.parquet — 17396 строк, 32 столбцов
test_scaled.parquet — 17394 строк, 32 столбцов
train_unscaled.parquet — 17396 строк, 32 столбцов
test_unscaled.parquet — 17394 строк, 32 столбцов
Сохранено в папку: C:\Users\17114\Desktop\data
